# Protein Prep

Recommend settings and prepare a protein with one `ProteinPrep` object.
`recommend()` returns a component table and does not bind an execution ID.
`run()` is blocking and requires `model_missing_loops=False`; `start()`
submits asynchronous preparation with loop modelling either off or on.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin.drug_discovery import BRD_DATA_DIR, Protein, ProteinPrep, StructureReport
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient.from_disk()
client

# Structure Report

In this section we demonstrate how we can generate a structure report on Protein inputs

## Structure Report from a file

Here, we run a structure report on a file

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
sr = StructureReport(protein=protein)
results = sr.run()

In [ ]:
results[0]

## Structure report from a PDB ID

We can also run a structure report for any entry in RCSB using a PDB ID. This structure report is generated without downloading the file, using metadata from RCSB

In [ ]:
sr = StructureReport(pdb_id="6GOG")
results = sr.run()
results[0]

In [ ]:
sr = StructureReport(pdb_id="1EW3")
results = sr.run()
results[0]

## Recommend and review

`recommend()` blocks and returns a table of components. Filter unresolved
decisions with `recommendation(decision="review")`. Resolve every `review`
before prepare.

In [ ]:
protein = Protein.from_pdb_id("1eby")
protein.show()

In [ ]:
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()

Now we can apply transformations using method chaining:

In [ ]:
(
    prep
    .keep(kind="water", subtype="coordinating")
    .skip(kind="water", subtype="crystal")
    .skip(decision="review")
)


## Run Protein prep

Now we have made our decision, we can run protein prep. 

In [ ]:
prep.model_missing_loops = False
prepared = prep.run()


In [ ]:
prepared.download()
prepared.show()

In [ ]:
from itertools import islice

with open(prepared.local_path) as f:
    print("".join(islice(f, 15)), end="")